# End-to-End BO Workflow: Sequential → Batch → Async

このNotebookは個別APIのカタログではなく、実運用でBO pipelineをどう組み立てるかを示します。

```text
completed observations
  → fit surrogate
  → construct acquisition
  → optimize acquisition
  → dispatch candidate(s)
  → evaluate / wait
  → update completed data
  → repeat
```

同じsurrogateを基準に、sequential、joint batch、asynchronous replacementへ段階的に切り替えます。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import (
    qLogExpectedImprovement,
    qLogNoisyExpectedImprovement,
)
from botorch.fit import fit_gpytorch_mll
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP
from robotorchan.optim import optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Problem and completed observations

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.25 * X


bounds = torch.tensor([[0.0], [1.0]])
train_X = torch.linspace(0.05, 0.95, 8).unsqueeze(-1)
train_Y = objective(train_X)

print("completed observations:", train_X.shape[0])

## 3. A small public-API fitting function

Workflow内の重複を減らすための局所helperです。robotorchan専用frameworkを新設する意図はありません。

In [ ]:
def fit_model(X: torch.Tensor, Y: torch.Tensor) -> SingleTaskGP:
    model = SingleTaskGP(X, Y)
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    return model

## 4. Sequential BO: fit → acquisition → candidate → evaluate → append

In [ ]:
for iteration in range(2):
    model = fit_model(train_X, train_Y)
    acquisition = qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
        sampler=SobolQMCNormalSampler(torch.Size([64]), seed=100 + iteration),
    )
    candidate, _ = optimize_acqf(
        acq_function=acquisition,
        bounds=bounds,
        q=1,
        optimizer="botorch",
        num_restarts=3,
        raw_samples=64,
    )
    candidate = candidate.detach()
    observed_Y = objective(candidate)
    train_X = torch.cat([train_X, candidate], dim=0)
    train_Y = torch.cat([train_Y, observed_Y], dim=0)
    print(iteration, candidate.item(), observed_Y.item())

## 5. Joint batch BO: q is a joint decision

`q=2`はq=1 optimizerを2回呼ぶことと同じではありません。MC acquisitionは2候補のjoint posterior utilityを評価します。

In [ ]:
model = fit_model(train_X, train_Y)
batch_acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=201),
)
batch_X, _ = optimize_acqf(
    acq_function=batch_acquisition,
    bounds=bounds,
    q=2,
    optimizer="botorch",
    num_restarts=3,
    raw_samples=64,
)
batch_X = batch_X.detach()
batch_Y = objective(batch_X)

assert batch_X.shape == torch.Size([2, 1])
train_X = torch.cat([train_X, batch_X], dim=0)
train_Y = torch.cat([train_Y, batch_Y], dim=0)
print("joint batch:\n", batch_X)

## 6. Async BO: completed data and pending work are different states

2 workersが実行中で、その間にreplacement candidateを1点生成する状況を作ります。pending pointsはtraining dataへ追加しません。

In [ ]:
X_pending = torch.tensor([[0.22], [0.78]])
completed_rows = train_X.shape[0]
model = fit_model(train_X, train_Y)
async_acquisition = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    X_pending=X_pending,
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=301),
    prune_baseline=False,
)
replacement_X, _ = optimize_acqf(
    acq_function=async_acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=3,
    raw_samples=64,
)

assert train_X.shape[0] == completed_rows
assert replacement_X.shape == torch.Size([1, 1])
print("pending workers:\n", X_pending)
print("replacement candidate:", replacement_X.detach())

## 7. Complete one worker, keep the other pending, and continue

完了した評価だけをtraining dataへ追加します。未完了点は次のacquisitionでも`X_pending`として残します。

In [ ]:
completed_X = X_pending[:1]
remaining_pending = X_pending[1:]
completed_Y = objective(completed_X)
train_X = torch.cat([train_X, completed_X], dim=0)
train_Y = torch.cat([train_Y, completed_Y], dim=0)

model = fit_model(train_X, train_Y)
next_acquisition = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    X_pending=remaining_pending,
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=302),
    prune_baseline=False,
)
next_X, _ = optimize_acqf(
    acq_function=next_acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=3,
    raw_samples=64,
)

print("new completed rows:", train_X.shape[0])
print("still pending:", remaining_pending)
print("next candidate:", next_X.detach())

## 8. What changes across workflows?

| Workflow | Training data | Acquisition context | Candidate semantics |
| --- | --- | --- | --- |
| Sequential | completed only | none | q=1 |
| Joint batch | completed only | none | q>1 joint utility |
| Async | completed only | `X_pending` | replacement candidate(s) |

Pending points、fantasy-model batch、one-shot auxiliary rowsはすべて別概念です。

## 9. Choose the specialized workflow by problem structure

| Need | Notebook |
| --- | --- |
| first standard BO loop | `01_single_task_gp.ipynb` |
| mixed continuous/categorical | `02_mixed_single_task_gp.ipynb` |
| multi-fidelity / cost-aware | `03_multi_fidelity_gp.ipynb` |
| MultiTask / Kronecker | `04_multitask_gp.ipynb` |
| high-dimensional reduction | `19_reduced_gp.ipynb` |
| multi-objective | `27_multiobjective_acquisition.ipynb` |
| regression active learning | `28_active_learning_acquisition.ipynb` |
| objective / posterior transform | `29_objective_posterior_transform.ipynb` |
| sampling contract | `30_posterior_sampling.ipynb` |
| optimizer / constraints | `31_acquisition_optimization.ipynb` |
| batch / async / fantasy | `32_batch_async_fantasization.ipynb` |
| initialization | `33_acquisition_initialization.ipynb` |
| TuRBO | `34_turbo_trust_region.ipynb` |
| robust input perturbation | `35_robust_input_perturbation.ipynb` |

## 10. Cross-layer checklist

実運用Workflowでは、モデル名だけでなく次を確認します。

1. training dataのshapeと構造列
2. posterior / sampler contract
3. objective / posterior transform semantics
4. acquisitionがsingle, batch, noisy, MO, MF, ALのどれか
5. candidate constraintsとoutcome constraintsの区別
6. initialization shapeとone-shot例外
7. optimizerがvariable typeとq semanticsを扱えるか
8. pending / fantasy lifecycle
9. candidateがどの座標系で返るか
10. evaluation後に何をtraining dataとstateへ戻すか

## 11. Related documentation

- `docs/development/notebook-executable-documentation-architecture.md`
- `docs/development/notebook-authoring-standard.md`
- `docs/optimization/acquisition-integration.md`
- `docs/optimization/batch-async-fantasization.md`

このNotebookを入口にし、問題構造が増えた時だけ専門Workflowへ分岐するのが推奨構成です。